In [223]:
from binance.um_futures import UMFutures
from database.adatabase import ADatabase
from binance_parameter_creator.binance_parameter_creator import BinanceParameterCreator as bpc
from binance_strategy.binance_strategy_factory import BinanceStrategyFactory
from crypto_parameter.acrypto_parameter import ACryptoParameter
from time import sleep
import pandas as pd
from datetime import datetime, timedelta
from binance.um_futures import UMFutures
from database.adatabase import ADatabase
from time import sleep
import pandas as pd
import random
import matplotlib.pyplot as plt
import pytz
kst_timezone = pytz.timezone('Asia/Seoul')

In [224]:
db = ADatabase("sapling")
db.cloud_connect()
bots = db.retrieve("crypto_bots")
keys = db.retrieve("crypto_secrets")
parameters = db.retrieve("crypto_parameter")
db.disconnect()
## parameters
for bot in bots.iterrows():
    user = bot[1]["username"]
    live = bot[1]["live"]
    if live == True:
        parameter = parameters[parameters["username"]==user].to_dict("records")[0]
        ticker = parameter["ticker"]
        secret = keys[keys["username"]==user]["bsecret"].item()
        key = keys[keys["username"]==user]["bkey"].item()
        umf = UMFutures(key,secret)
        account = umf.account()
        trades = umf.get_account_trades("XRPUSDT")
        trades = pd.DataFrame(trades)
        print(account["totalUnrealizedProfit"])
        positions = pd.DataFrame(account["positions"])
        xrp_positions = positions[positions["symbol"]==ticker]
        orders = pd.DataFrame(umf.get_all_orders("XRPUSDT"))
        new_orders = orders[orders["status"]=="NEW"]
        

-0.03573045


In [225]:
xrp_positions

,symbol,initialMargin,maintMargin,unrealizedProfit,positionInitialMargin,openOrderInitialMargin,leverage,isolated,entryPrice,breakEvenPrice,maxNotional,positionSide,positionAmt,notional,isolatedWallet,updateTime,bidNotional,askNotional
122,XRPUSDT,4.94788227,0.37109115,-0.03573045,4.94788227,0,15,True,0.5495,0.54922525,750000,BOTH,-135.0,-74.21823045,4.90840899,1708993149992,0,0


In [226]:
new_orders

,orderId,symbol,status,clientOrderId,price,avgPrice,origQty,executedQty,cumQuote,timeInForce,...,workingType,priceMatch,selfTradePreventionMode,goodTillDate,priceProtect,origType,time,updateTime,activatePrice,priceRate


In [227]:
trades["date"] = [datetime.utcfromtimestamp(float(x) // 1000.0).replace(tzinfo=pytz.utc).astimezone(kst_timezone) for x in trades["time"]]

In [228]:
trades["realizedPnl"] = [float(x) for x in trades["realizedPnl"]]
trades["commission"] = [float(x) for x in trades["commission"]]
trades["price"] = [float(x) for x in trades["price"]]
trades["w/l"] = ["W" if x > 0 else "L" if x < 0 else "N" for x in trades["realizedPnl"]]
trades = trades[trades["date"]>datetime(2024,2,22).replace(tzinfo=pytz.utc).astimezone(kst_timezone)]
trades = trades.groupby(["date","w/l"]).agg({"price":"mean","commission":"sum","realizedPnl":"sum"}).reset_index()
trades["agg_pnl"] = trades["realizedPnl"].cumsum()
trades["agg_commission"] = trades["commission"].cumsum()
trades["pnl"] = trades["agg_pnl"] - trades["agg_commission"]
trades["entry_price"] = trades["price"].shift(1)
trades["price_diff"] = trades["price"].pct_change() * 100
# trades = trades[trades["w/l"]!="N"]
trades[["date","price","price_diff","w/l","realizedPnl","agg_pnl","commission","agg_commission","pnl"]]

,date,price,price_diff,w/l,realizedPnl,agg_pnl,commission,agg_commission,pnl
0,2024-02-22 13:22:51+09:00,0.5434,NaN,N,0.00000,0.00000,0.222549,0.222549,-0.222549
1,2024-02-22 13:23:15+09:00,0.5435,0.018403,W,0.08191,0.08191,0.222590,0.445140,-0.363230
2,2024-02-22 13:43:26+09:00,0.5419,-0.294388,N,0.00000,0.08191,0.014902,0.460042,-0.378132
3,2024-02-22 13:43:50+09:00,0.5419,0.000000,N,0.00000,0.08191,0.014902,0.474944,-0.393034
4,2024-02-22 13:44:40+09:00,0.5419,0.000000,N,0.00000,0.08191,0.014902,0.489847,-0.407937
...,...,...,...,...,...,...,...,...,...
232,2024-02-27 09:00:56+09:00,0.5509,-0.072556,N,0.00000,-10.30439,0.033054,25.709678,-36.014068
233,2024-02-27 09:01:06+09:00,0.5512,0.054456,L,-0.03600,-10.34039,0.033072,25.742750,-36.083140
234,2024-02-27 09:01:14+09:00,0.5512,0.000000,N,0.00000,-10.34039,0.033072,25.775822,-36.116212
235,2024-02-27 09:19:02+09:00,0.5495,-0.308418,W,0.20400,-10.13639,0.032970,25.808792,-35.945182
